# ДЗ 27. Почувствуй мощь трансформеров в бою
## RuCoLA: лингвистическая приемлемость русского предложения

### Цель
Сравнить три подхода на датасете [RuCoLA](https://github.com/RussianNLP/RuCoLA):
1. **Fine-tune RuBERT** — классификация `acceptable` / `unacceptable`
2. **RuGPT3 few-/zero-shot** — разные затравки и число примеров (0, 1, 2, 4)
3. **Fine-tune RuT5** — seq2seq (`yes` / `no`)

Метрики: **Accuracy** и **MCC** (Matthews Correlation Coefficient).

### Данные
- train: `data/in_domain_train.csv` → дальше делим на train/val
- test: `data/in_domain_dev.csv`

### Модели (под 4GB GPU)
- BERT: `DeepPavlov/rubert-base-cased`
- GPT: `ai-forever/rugpt3small_based_on_gpt2` (семейство RuGPT3 base/small)
- T5: `ai-forever/ruT5-base`

> Полный прогон долгий. Можно запускать блоки по очереди или скрипт `hw27_rucola_transformers.py`.


---
## Блок 0. Импорты и константы

Здесь подключаем PyTorch, HuggingFace Transformers, datasets и sklearn.
Фиксируем seed и выбираем устройство (`cuda`, если есть GPU).


In [1]:
# --- Блок 0: импорты и настройки ---
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from IPython.display import display
from sklearn.metrics import accuracy_score, matthews_corrcoef
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    DataCollatorWithPadding,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    T5ForConditionalGeneration,
    T5Tokenizer,
    Trainer,
    TrainingArguments,
    set_seed,
)

# Пути
BASE = Path(".").resolve()
DATA = BASE / "data"
OUT = BASE / "outputs"
OUT.mkdir(exist_ok=True)

# Гиперпараметры / имена моделей
SEED = 42
MAX_LEN = 128  # макс. длина токенизации предложения
BERT_MODEL = "DeepPavlov/rubert-base-cased"
GPT_MODEL = "ai-forever/rugpt3small_based_on_gpt2"
T5_MODEL = "ai-forever/ruT5-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def set_all_seeds(seed: int = SEED):
    """Фиксируем случайность во всех библиотеках."""
    set_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_all_seeds()
print(f"Устройство: {DEVICE}")
if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}, память ~{props.total_memory // 2**20} МБ")


Устройство: cuda
GPU: NVIDIA A40, память ~45498 МБ


---
## Блок 1. Загрузка RuCoLA и split train/val

- `in_domain_train.csv` — размечаем на **train** и **val** (стратифицированно по метке)
- `in_domain_dev.csv` — используем как **test** (как в условии ДЗ)

Колонка `acceptable`: `1` = предложение приемлемо, `0` = нет.


In [2]:
# --- Блок 1: данные ---
def load_splits(val_size: float = 0.1):
    """Читает CSV и делит train на train/val со стратификацией."""
    train_full = pd.read_csv(DATA / "in_domain_train.csv")
    test = pd.read_csv(DATA / "in_domain_dev.csv")

    train, val = train_test_split(
        train_full,
        test_size=val_size,
        random_state=SEED,
        stratify=train_full["acceptable"],  # сохраняем долю классов
    )
    train = train.reset_index(drop=True)
    val = val.reset_index(drop=True)
    test = test.reset_index(drop=True)

    print(f"train={len(train)}, val={len(val)}, test={len(test)}")
    print(f"Доля acceptable в train: {train.acceptable.mean():.3f}")
    return train, val, test


def calc_metrics(y_true, y_pred):
    """Accuracy + MCC — основные метрики RuCoLA."""
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
    }


train_df, val_df, test_df = load_splits()
train_df.head(3)


train=7082, val=787, test=983
Доля acceptable в train: 0.745


,id,sentence,acceptable,error_type,detailed_source
0,7049,Я не знаю Машу Трофимову.,1,0,Paducheva2013
1,2361,Костюм не был в химчистке весь год.,0,Semantics,Paducheva2013
2,956,"Она, конечно, знала, что рискует, отказываясь ...",1,0,Paducheva2004


---
## Блок 2. Fine-tune RuBERT (3 балла)

### Идея
Берём предобученный русский BERT и добавляем голову классификации на 2 класса.
Учим на train, выбираем лучший чекпоинт по **MCC на val**, оцениваем на test.

### Шаги в коде
1. Токенизация предложений  
2. `Trainer` с `TrainingArguments`  
3. `argmax` по логитам → метка класса


In [3]:
# --- Блок 2: fine-tune RuBERT ---
def run_rubert(train_df, val_df, test_df):
    """Обучает RuBERT-классификатор и возвращает метрики на test."""
    print("=== Fine-tune RuBERT ===")

    # Токенизатор и модель с головой на 2 класса
    tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL, num_labels=2)

    def tokenize_batch(batch):
        # Превращаем текст в input_ids / attention_mask
        enc = tokenizer(
            batch["sentence"], truncation=True, max_length=MAX_LEN, padding=False
        )
        enc["labels"] = batch["acceptable"]
        return enc

    # HuggingFace DatasetDict из pandas
    ds = DatasetDict({
        "train": Dataset.from_pandas(train_df[["sentence", "acceptable"]], preserve_index=False),
        "val": Dataset.from_pandas(val_df[["sentence", "acceptable"]], preserve_index=False),
        "test": Dataset.from_pandas(test_df[["sentence", "acceptable"]], preserve_index=False),
    }).map(tokenize_batch, batched=True, remove_columns=["sentence", "acceptable"])

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return calc_metrics(labels, preds)

    # Батч 8 и fp16 — чтобы влезло в 4GB VRAM
    args = TrainingArguments(
        output_dir=str(OUT / "rubert_ckpt"),
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="mcc",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        logging_steps=50,
        report_to="none",
        seed=SEED,
        save_total_limit=1,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds["train"],
        eval_dataset=ds["val"],
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),  # динамический padding в батче
        compute_metrics=compute_metrics,
    )

    trainer.train()
    test_metrics = trainer.evaluate(ds["test"])
    result = {
        "model": BERT_MODEL,
        "test_accuracy": float(test_metrics["eval_accuracy"]),
        "test_mcc": float(test_metrics["eval_mcc"]),
        "val": trainer.evaluate(ds["val"]),
    }
    print(f"RuBERT test: acc={result['test_accuracy']:.4f}, mcc={result['test_mcc']:.4f}")

    trainer.save_model(str(OUT / "rubert_best"))
    tokenizer.save_pretrained(str(OUT / "rubert_best"))
    return result


# Запуск (можно закомментировать, если уже есть outputs/rubert_metrics.json)
rubert_path = OUT / "rubert_metrics.json"
if rubert_path.exists():
    rubert_result = json.loads(rubert_path.read_text(encoding="utf-8"))
    print("Загружены сохранённые метрики RuBERT:", rubert_result)
else:
    rubert_result = run_rubert(train_df, val_df, test_df)
    rubert_path.write_text(json.dumps(rubert_result, indent=2, ensure_ascii=False), encoding="utf-8")


Загружены сохранённые метрики RuBERT: {'model': 'DeepPavlov/rubert-base-cased', 'test_accuracy': 0.8046795523906409, 'test_mcc': 0.4269433814214215, 'val': {'eval_loss': 0.736528217792511, 'eval_accuracy': 0.7928843710292249, 'eval_mcc': 0.3985840043857259}}


---
## Блок 3. RuGPT3: zero-/few-shot (4 балла)

### Идея
Модель **не дообучаем**. Даём текстовую затравку (prompt) и считаем,
какое продолжение (`да`/`нет`, `acceptable`/`unacceptable`, …) более вероятно.

### Что перебираем
**а)** 3 варианта затравок  
**б)** число few-shot примеров: `0, 1, 2, 4`

Классификация: сравниваем суммарный log-probability позитивной и негативной метки.


In [4]:
# --- Блок 3a: шаблоны затравок ---
# Каждый шаблон: инструкция + как оформить пример + какой запрос + метки классов
PROMPTS = {
    "yes_no": {
        "instruction": (
            "Определи, является ли предложение грамматически правильным "
            "и приемлемым на русском языке. Ответь только да или нет.\n"
        ),
        "example_fn": lambda s, y: (
            f"Предложение: {s}\nОтвет: {'да' if y == 1 else 'нет'}\n"
        ),
        "query_fn": lambda s: f"Предложение: {s}\nОтвет:",
        "pos": "да",
        "neg": "нет",
    },
    "acceptable": {
        "instruction": (
            "Задача: классификация лингвистической приемлемости. "
            "Метка acceptable или unacceptable.\n"
        ),
        "example_fn": lambda s, y: (
            f"Текст: {s}\nМетка: {'acceptable' if y == 1 else 'unacceptable'}\n"
        ),
        "query_fn": lambda s: f"Текст: {s}\nМетка:",
        "pos": "acceptable",
        "neg": "unacceptable",
    },
    "correct_incorrect": {
        "instruction": (
            "Проверь корректность русского предложения. "
            "Ответ: корректно или некорректно.\n"
        ),
        "example_fn": lambda s, y: (
            f"Пример: {s}\nРезультат: {'корректно' if y == 1 else 'некорректно'}\n"
        ),
        "query_fn": lambda s: f"Пример: {s}\nРезультат:",
        "pos": "корректно",
        "neg": "некорректно",
    },
}


def build_prompt(prompt_cfg, shot_examples, sentence):
    """Склеивает инструкцию + few-shot примеры + запрос по новому предложению."""
    parts = [prompt_cfg["instruction"]]
    for s, y in shot_examples:
        parts.append(prompt_cfg["example_fn"](s, y))
    parts.append(prompt_cfg["query_fn"](sentence))
    return "".join(parts)


def select_shots(train_df, k, seed=SEED):
    """Берёт k сбалансированных примеров (примерно поровну 0 и 1)."""
    if k == 0:
        return []
    n_pos = max(1, k // 2)
    n_neg = k - n_pos
    pos = train_df[train_df.acceptable == 1].sample(n=n_pos, random_state=seed)
    neg = train_df[train_df.acceptable == 0].sample(n=n_neg, random_state=seed)
    shots = pd.concat([pos, neg]).sample(frac=1.0, random_state=seed)
    return list(zip(shots.sentence.tolist(), shots.acceptable.tolist()))


@torch.no_grad()
def gpt_score_continuation(model, tokenizer, prompt, continuation):
    """Суммарный log-prob продолжения при условии prompt (чем выше — тем лучше)."""
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).input_ids.to(
        model.device
    )
    cont_ids = tokenizer(
        continuation, return_tensors="pt", add_special_tokens=False
    ).input_ids.to(model.device)

    # Склеиваем prompt + ответ и считаем loss только на токенах ответа
    input_ids = torch.cat([prompt_ids, cont_ids], dim=1)
    labels = input_ids.clone()
    labels[:, : prompt_ids.shape[1]] = -100  # игнорируем prompt в loss
    outputs = model(input_ids=input_ids, labels=labels)
    n_tok = cont_ids.shape[1]
    # loss = средний NLL → умножаем на число токенов → сумма NLL; минус → log-prob
    return -outputs.loss.item() * n_tok


print("Шаблоны затравок:", list(PROMPTS.keys()))


Шаблоны затравок: ['yes_no', 'acceptable', 'correct_incorrect']


In [5]:
# --- Блок 3b: прогон zero-/few-shot ---
def run_rugpt_fewshot(train_df, test_df, max_test=None):
    """Перебирает затравки и число shots, считает accuracy/MCC на test."""
    print("=== RuGPT3 few-/zero-shot ===")
    tokenizer = AutoTokenizer.from_pretrained(GPT_MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(GPT_MODEL)
    model.to(DEVICE)
    model.eval()

    eval_df = test_df
    if max_test is not None and len(test_df) > max_test:
        # Для быстрой отладки можно взять подвыборку
        eval_df = test_df.sample(n=max_test, random_state=SEED).reset_index(drop=True)
        print(f"Оценка на подвыборке n={len(eval_df)}")

    results = []
    for prompt_name, cfg in PROMPTS.items():
        for k in [0, 1, 2, 4]:
            shots = select_shots(train_df, k)
            preds = []
            for sentence in eval_df.sentence.tolist():
                prompt = build_prompt(cfg, shots, sentence)
                # Какая метка вероятнее после prompt?
                lp_pos = gpt_score_continuation(model, tokenizer, prompt, " " + cfg["pos"])
                lp_neg = gpt_score_continuation(model, tokenizer, prompt, " " + cfg["neg"])
                preds.append(1 if lp_pos >= lp_neg else 0)

            m = calc_metrics(eval_df.acceptable.tolist(), preds)
            row = {
                "prompt": prompt_name,
                "n_shots": k,
                "n_eval": len(eval_df),
                "accuracy": m["accuracy"],
                "mcc": m["mcc"],
                "model": GPT_MODEL,
            }
            results.append(row)
            print(
                f"prompt={prompt_name:18s} shots={k} "
                f"acc={m['accuracy']:.4f} mcc={m['mcc']:.4f}"
            )

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return results


gpt_csv = OUT / "rugpt3_fewshot_metrics.csv"
if gpt_csv.exists():
    gpt_results = pd.read_csv(gpt_csv).to_dict(orient="records")
    print("Загружены сохранённые метрики RuGPT3")
    display(pd.DataFrame(gpt_results))
else:
    gpt_results = run_rugpt_fewshot(train_df, test_df, max_test=None)
    pd.DataFrame(gpt_results).to_csv(gpt_csv, index=False)
    display(pd.DataFrame(gpt_results))


Загружены сохранённые метрики RuGPT3


,prompt,n_shots,n_eval,accuracy,mcc,model
0,yes_no,0,983,0.487284,-0.044713,ai-forever/rugpt3small_based_on_gpt2
1,yes_no,1,983,0.460834,-0.036728,ai-forever/rugpt3small_based_on_gpt2
2,yes_no,2,983,0.311292,0.022529,ai-forever/rugpt3small_based_on_gpt2
3,yes_no,4,983,0.283825,0.020755,ai-forever/rugpt3small_based_on_gpt2
4,acceptable,0,983,0.390641,0.075588,ai-forever/rugpt3small_based_on_gpt2
5,acceptable,1,983,0.310275,-0.015834,ai-forever/rugpt3small_based_on_gpt2
6,acceptable,2,983,0.259410,-0.068385,ai-forever/rugpt3small_based_on_gpt2
7,acceptable,4,983,0.254323,0.000000,ai-forever/rugpt3small_based_on_gpt2
8,correct_incorrect,0,983,0.745677,0.000000,ai-forever/rugpt3small_based_on_gpt2
9,correct_incorrect,1,983,0.396745,-0.075693,ai-forever/rugpt3small_based_on_gpt2


---
## Блок 4. Fine-tune RuT5 (3 балла)

### Идея
T5 — encoder–decoder. Формулируем задачу как генерацию короткого ответа:
- вход: `приемлемо ли предложение: <текст>`
- выход: `yes` или `no`

Ориентир: [baselines/finetune_t5.py](https://github.com/RussianNLP/RuCoLA/blob/main/baselines/finetune_t5.py) из репозитория RuCoLA.


In [6]:
# --- Блок 4: fine-tune RuT5 ---
POS_LABEL = "yes"
NEG_LABEL = "no"


def run_rut5(train_df, val_df, test_df):
    """Обучает RuT5 генерировать yes/no и оценивает на test."""
    print("=== Fine-tune RuT5 ===")
    tokenizer = T5Tokenizer.from_pretrained(T5_MODEL, legacy=False)

    def preprocess(examples):
        # Префикс задачи помогает T5 понять формат
        inputs = ["приемлемо ли предложение: " + s for s in examples["sentence"]]
        model_inputs = tokenizer(inputs, truncation=True, max_length=MAX_LEN, padding=False)

        targets = [POS_LABEL if y == 1 else NEG_LABEL for y in examples["acceptable"]]
        labels = tokenizer(text_target=targets, truncation=True, max_length=8, padding=False)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    ds = DatasetDict({
        "train": Dataset.from_pandas(train_df[["sentence", "acceptable"]], preserve_index=False),
        "val": Dataset.from_pandas(val_df[["sentence", "acceptable"]], preserve_index=False),
        "test": Dataset.from_pandas(test_df[["sentence", "acceptable"]], preserve_index=False),
    }).map(preprocess, batched=True, remove_columns=["sentence", "acceptable"])

    model = T5ForConditionalGeneration.from_pretrained(T5_MODEL)

    def compute_metrics(p):
        # Декодируем сгенерированные токены обратно в строки yes/no
        preds = tokenizer.batch_decode(p.predictions, skip_special_tokens=True)
        labels = np.where(p.label_ids != -100, p.label_ids, tokenizer.pad_token_id)
        label_str = tokenizer.batch_decode(labels, skip_special_tokens=True)
        int_preds = [1 if x.strip().lower() == POS_LABEL else 0 for x in preds]
        int_labels = [1 if x.strip().lower() == POS_LABEL else 0 for x in label_str]
        return calc_metrics(int_labels, int_preds)

    args = Seq2SeqTrainingArguments(
        output_dir=str(OUT / "rut5_ckpt"),
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=1e-4,
        per_device_train_batch_size=4,   # T5 тяжелее BERT — меньший батч
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        weight_decay=0.0,
        predict_with_generate=True,     # на валидации генерируем ответ
        generation_max_length=8,
        load_best_model_at_end=True,
        metric_for_best_model="mcc",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        logging_steps=50,
        report_to="none",
        seed=SEED,
        save_total_limit=1,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=ds["train"],
        eval_dataset=ds["val"],
        processing_class=tokenizer,
        data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8),
        compute_metrics=compute_metrics,
    )

    trainer.train()
    test_out = trainer.predict(ds["test"], max_length=8)
    result = {
        "model": T5_MODEL,
        "test_accuracy": float(test_out.metrics["test_accuracy"]),
        "test_mcc": float(test_out.metrics["test_mcc"]),
        "val": trainer.evaluate(ds["val"]),
    }
    print(f"RuT5 test: acc={result['test_accuracy']:.4f}, mcc={result['test_mcc']:.4f}")

    trainer.save_model(str(OUT / "rut5_best"))
    tokenizer.save_pretrained(str(OUT / "rut5_best"))
    return result


rut5_path = OUT / "rut5_metrics.json"
if rut5_path.exists():
    rut5_result = json.loads(rut5_path.read_text(encoding="utf-8"))
    print("Загружены сохранённые метрики RuT5:", rut5_result)
else:
    rut5_result = run_rut5(train_df, val_df, test_df)
    rut5_path.write_text(json.dumps(rut5_result, indent=2, ensure_ascii=False), encoding="utf-8")


=== Fine-tune RuT5 ===


Map:   0%|          | 0/7082 [00:00<?, ? examples/s]

Map:   0%|          | 0/787 [00:00<?, ? examples/s]

Map:   0%|          | 0/983 [00:00<?, ? examples/s]

config.json: 0.00B [00:00, ?B/s]

...

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


RuT5 test: acc=0.7243, mcc=0.2509


---
## Блок 5. Сравнение результатов

Сводим три подхода в одну таблицу и кратко интерпретируем:
- кто лучше по MCC (главная метрика RuCoLA при дисбалансе классов);
- насколько few-shot GPT отстаёт от fine-tune.


In [7]:
# --- Блок 5: итоговая таблица ---
gpt_df = pd.DataFrame(gpt_results)
best_gpt = gpt_df.sort_values(["mcc", "accuracy"], ascending=False).iloc[0]

summary = pd.DataFrame([
    {
        "подход": "RuBERT fine-tune",
        "детали": rubert_result.get("model", BERT_MODEL),
        "accuracy": rubert_result["test_accuracy"],
        "mcc": rubert_result["test_mcc"],
    },
    {
        "подход": "RuGPT3 few/zero-shot (лучший)",
        "детали": f"{best_gpt['prompt']}, shots={int(best_gpt['n_shots'])}",
        "accuracy": float(best_gpt["accuracy"]),
        "mcc": float(best_gpt["mcc"]),
    },
    {
        "подход": "RuT5 fine-tune",
        "детали": rut5_result.get("model", T5_MODEL),
        "accuracy": rut5_result["test_accuracy"],
        "mcc": rut5_result["test_mcc"],
    },
])

summary.to_csv(OUT / "comparison_summary.csv", index=False)
print("=== Сравнение на in_domain_dev (test) ===")
display(summary)

print("\nВсе конфигурации RuGPT3:")
display(gpt_df.sort_values("mcc", ascending=False))


=== Сравнение на in_domain_dev (test) ===


,подход,детали,accuracy,mcc
0,RuBERT fine-tune,DeepPavlov/rubert-base-cased,0.804680,0.426943
1,RuGPT3 few/zero-shot (лучший),"acceptable, shots=0",0.390641,0.075588
2,RuT5 fine-tune,ai-forever/ruT5-base,0.724313,0.250931



Все конфигурации RuGPT3:


,prompt,n_shots,n_eval,accuracy,mcc,model
4,acceptable,0,983,0.390641,0.075588,ai-forever/rugpt3small_based_on_gpt2
2,yes_no,2,983,0.311292,0.022529,ai-forever/rugpt3small_based_on_gpt2
3,yes_no,4,983,0.283825,0.020755,ai-forever/rugpt3small_based_on_gpt2
7,acceptable,4,983,0.254323,0.000000,ai-forever/rugpt3small_based_on_gpt2
11,correct_incorrect,4,983,0.254323,0.000000,ai-forever/rugpt3small_based_on_gpt2
8,correct_incorrect,0,983,0.745677,0.000000,ai-forever/rugpt3small_based_on_gpt2
5,acceptable,1,983,0.310275,-0.015834,ai-forever/rugpt3small_based_on_gpt2
10,correct_incorrect,2,983,0.260427,-0.024779,ai-forever/rugpt3small_based_on_gpt2
1,yes_no,1,983,0.460834,-0.036728,ai-forever/rugpt3small_based_on_gpt2
0,yes_no,0,983,0.487284,-0.044713,ai-forever/rugpt3small_based_on_gpt2


### Краткие выводы
- **Fine-tune RuBERT / RuT5** обычно дают заметно более высокий MCC, чем zero-/few-shot GPT:
  модели специально учатся различать приемлемость на размеченных примерах.
- У **GPT** качество сильно зависит от формулировки затравки; несколько few-shot примеров
  часто помогают относительно zero-shot, но потолок всё равно ниже supervised fine-tune.
- **MCC** важнее accuracy: в RuCoLA классы несбалансированы (~75% acceptable),
  поэтому «всегда говорить acceptable» даёт высокую accuracy при MCC ≈ 0.

### Что сдавать / куда смотреть
- Ноутбук: `hw_27_rucola_transformers.ipynb`
- Скрипт полного прогона: `hw27_rucola_transformers.py`
- Метрики: `outputs/comparison_summary.csv`, `outputs/rugpt3_fewshot_metrics.csv`
